# Unidad 2: Aprendizaje Automático 
## Importancia de Variables
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Importantes](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-izafi-27098358.jpg)

[Foto de Doğan Alpaslan Demir](https://www.pexels.com/es-es/foto/juego-partido-ajedrez-estrategia-27098358/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/16_FeatureImportances.ipynb)


## 🎯 ¿Qué vamos a aprender?

Random Forest no solo clasifica bien: también nos dice **qué variables son más relevantes** para hacer sus predicciones. Esta capacidad se llama **Feature Importance** (importancia de variables).

Al finalizar, vas a poder:
- ✅ Entender cómo Random Forest calcula la importancia de cada variable
- ✅ Visualizar las **Feature Importances** en un gráfico de barras
- ✅ Seleccionar un subconjunto de features y reentrenar el modelo
- ✅ Comparar el rendimiento del modelo **completo** vs el modelo con **features seleccionadas**
- ✅ Comprender el trade-off entre **simplicidad** y **rendimiento**

---

## 🧠 Conceptos Clave

### 🔑 ¿Qué es Feature Importance?

Random Forest calcula la importancia de cada feature midiendo **cuánto mejora la pureza** de los nodos cuando esa feature se usa para dividir, promediado sobre todos los árboles del bosque.

Formalmente, para cada feature $j$:

$$\text{Importancia}(j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{\text{nodo } k \text{ que usa } j} \frac{n_k}{n} \cdot \Delta\text{Gini}(k)$$

Donde:
- $T$ = número de árboles
- $n_k$ = muestras en el nodo $k$
- $n$ = total de muestras
- $\Delta\text{Gini}(k)$ = reducción en impureza Gini al dividir en $k$

Los valores se normalizan para que **sumen 1.0**.

### 🛠️ ¿Para qué sirve?

Feature Importance tiene múltiples aplicaciones:

| Aplicación | Descripción |
|-----------|-------------|
| 📉 **Feature Selection** | Eliminar variables irrelevantes → modelo más simple y rápido |
| 🔬 **Interpretabilidad** | Entender qué factores influyen en la predicción |
| 🧹 **Limpieza de datos** | Identificar variables redundantes |
| 🏥 **Contexto clínico** | En medicina: saber qué mediciones son realmente diagnósticas |

> ⚠️ **Limitación importante:** Feature Importance de Random Forest puede sobreestimar la importancia de variables con muchos valores únicos (variables continuas de alta cardinalidad). Para una alternativa más robusta, considerar *Permutation Importance*.

> 📌 **Referencias:**
> - Breiman, L. (2001). [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324). *Machine Learning*, 45, 5–32.
> - Strobl, C., et al. (2007). [Bias in random forest variable importance measures](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-8-25). *BMC Bioinformatics*, 8:25.
> - Scikit-learn: [feature_importances_](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.feature_importances_)
> - Scikit-learn: [Permutation Importance](https://scikit-learn.org/stable/modules/permutation_importance.html)

---

## 📦 Paso 1: Importar las Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics

print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar el Dataset y Entrenar el Modelo Completo

In [ ]:
# 📥 Cargar el dataset
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data['data'], columns=cancer_data['feature_names'])
df['target'] = cancer_data['target']

X = df[cancer_data.feature_names].values
y = df['target'].values

# ✂️ División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=52)

# 🌲 Modelo con todas las features
rf = RandomForestClassifier(random_state=52)
rf.fit(X_train, y_train)

print(f'📐 Dataset: {X.shape[0]} muestras × {X.shape[1]} features')
print(f'\n🌲 Modelo entrenado con TODAS las {X.shape[1]} features:')
print(f'   Accuracy: {rf.score(X_test, y_test):.4f}')

## 🔑 Paso 3: Calcular y Visualizar Feature Importances

Accedemos al atributo `.feature_importances_` del modelo entrenado, que contiene un array con la importancia de cada variable.

In [ ]:
# 🔑 Calcular Feature Importances
ft_imp = pd.Series(
    rf.feature_importances_,
    index=cancer_data.feature_names
).sort_values(ascending=False)

print('🔑 Feature Importances (ordenadas de mayor a menor):')
print(ft_imp.round(4).to_string())
print(f'\n✅ Suma total: {ft_imp.sum():.4f} (siempre debe ser ≈ 1.0)')

In [ ]:
# 📊 Visualización: gráfico de barras horizontales
fig, ax = plt.subplots(figsize=(9, 10))

colores = ['forestgreen' if i < 5 else 'steelblue' if i < 15 else 'lightgray'
           for i in range(len(ft_imp))]

bars = ax.barh(ft_imp.index, ft_imp.values, color=colores, edgecolor='black', alpha=0.85)

# Agregar valores en las barras
for bar, val in zip(bars, ft_imp.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8)

ax.invert_yaxis()  # La más importante arriba
ax.set_xlabel('Importancia relativa', fontsize=12)
ax.set_title('Feature Importances — Random Forest (Breast Cancer)', fontsize=13)

# Leyenda de colores
from matplotlib.patches import Patch
leyenda = [
    Patch(color='forestgreen', label='Top 5 (más importantes)'),
    Patch(color='steelblue',   label='Siguientes 10'),
    Patch(color='lightgray',   label='Menos relevantes'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## ✂️ Paso 4: Selección de Features

Vamos a seleccionar únicamente las **18 variables más importantes** (de 30 en total) y comparar si el modelo más simple mantiene el rendimiento.

> 💡 **¿Por qué 18?** Es una decisión de diseño: queremos reducir la dimensionalidad sin perder demasiada información. Podemos explorar diferentes umbrales para encontrar el punto óptimo de simplificación.

In [ ]:
# 🔍 Top-18 features más importantes
N_FEATURES = 18
selcols = ft_imp.index[:N_FEATURES]

print(f'✅ Features seleccionadas (Top-{N_FEATURES}):')
for i, col in enumerate(selcols, 1):
    imp = ft_imp[col]
    bar = '█' * int(imp * 100)
    print(f'  {i:2d}. {col:<35s} {imp:.4f} {bar}')

print(f'\n📊 Importancia acumulada (Top-{N_FEATURES}): {ft_imp.iloc[:N_FEATURES].sum():.4f}')

## 🤖 Paso 5: Reentrenar con Features Seleccionadas

In [ ]:
# 📥 Nuevo dataset con solo las features seleccionadas
X_sel = df[selcols]

X_trainb, X_testb, y_trainb, y_testb = train_test_split(X_sel, y, random_state=52)

# 🌲 Modelo con features seleccionadas
rfb = RandomForestClassifier(random_state=52)
rfb.fit(X_trainb, y_trainb)

print('✅ Modelo con features seleccionadas entrenado!')

## 📊 Paso 6: Comparar Modelos

In [ ]:
def get_metricas(modelo, X_te, y_te):
    y_p = modelo.predict(X_te)
    return {
        'Accuracy':  metrics.accuracy_score(y_te, y_p),
        'Precision': metrics.precision_score(y_te, y_p),
        'Recall':    metrics.recall_score(y_te, y_p),
        'F1':        metrics.f1_score(y_te, y_p),
    }

m_full = get_metricas(rf,  X_test,  y_test)
m_sel  = get_metricas(rfb, X_testb, y_testb)

print('=' * 65)
print('         📊 COMPARACIÓN: MODELO COMPLETO vs REDUCIDO')
print('=' * 65)
print(f'{"Métrica":<15} {"Modelo completo (30 feat)":>25} {"Modelo reducido ("+str(N_FEATURES)+" feat)":>20}')
print('-' * 65)
for metrica in ['Accuracy', 'Precision', 'Recall', 'F1']:
    v_full = m_full[metrica]
    v_sel  = m_sel[metrica]
    diff   = v_sel - v_full
    signo  = f'+{diff:.4f}' if diff >= 0 else f'{diff:.4f}'
    emoji  = '✅' if diff >= 0 else '⚠️'
    print(f'{metrica:<15} {v_full:>25.4f} {v_sel:>15.4f}   {emoji} ({signo})')
print('=' * 65)

In [ ]:
# 📊 Visualización comparativa
metric_names = list(m_full.keys())
vals_full = list(m_full.values())
vals_sel  = list(m_sel.values())

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, vals_full, width, label=f'Todas las features (30)',
       color='steelblue', edgecolor='black')
ax.bar(x + width/2, vals_sel,  width, label=f'Top-{N_FEATURES} features',
       color='forestgreen', edgecolor='black')

# Anotar diferencias
for i, (vf, vs) in enumerate(zip(vals_full, vals_sel)):
    diff = vs - vf
    color = 'green' if diff >= 0 else 'red'
    ax.annotate(f'{diff:+.3f}', xy=(i + width/2, vs),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=9, color=color, fontweight='bold')

ax.set_ylabel('Score')
ax.set_title(f'Comparación: Todas las Features vs Top-{N_FEATURES}', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0.90, 1.03)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 🔬 Paso 7 (Opcional): Explorar el Umbral de Selección

¿Cuántas features necesitamos para mantener un buen rendimiento? Entrenamos modelos con diferente número de features.

In [ ]:
# 🔬 Accuracy vs número de features seleccionadas
accs = []
n_range = range(1, len(cancer_data.feature_names) + 1)

for n in n_range:
    cols_n = ft_imp.index[:n]
    X_n = df[cols_n]
    Xtr, Xte, ytr, yte = train_test_split(X_n, y, random_state=52)
    m = RandomForestClassifier(random_state=52)
    m.fit(Xtr, ytr)
    accs.append(m.score(Xte, yte))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_range, accs, color='steelblue', marker='o', markersize=4, linewidth=1.5)
ax.axvline(x=N_FEATURES, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Selección actual: Top-{N_FEATURES}')
ax.set_xlabel('Número de features (Top-N)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy en función del número de features seleccionadas', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, len(cancer_data.feature_names) + 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_n_feat = int(np.argmax(accs)) + 1
print(f'💡 Número óptimo de features: {best_n_feat} (Accuracy={max(accs):.4f})')

## 🏁 Conclusiones

En este notebook aprendimos:

1. 🔑 **Feature Importance** mide cuánto contribuye cada variable a reducir la impureza en los árboles del bosque.
2. 📊 La visualización tipo **barras horizontales** ordenadas permite identificar rápidamente las variables más relevantes.
3. ✂️ Es posible **simplificar el modelo** eliminando features de baja importancia sin perder (a veces incluso ganando) rendimiento.
4. ⚠️ Feature Importance tiene **limitaciones**: puede sobrevaluar variables continuas. Para análisis más rigurosos, usar *Permutation Importance*.
5. 📈 Explorar el **umbral de selección** (¿cuántas features necesito?) es un proceso empírico valioso para optimizar el modelo.

### 🎓 ¡Hemos cubierto el ciclo completo de Random Forest!

| Notebook | Tema |
|----------|------|
| 12 | Introducción y comparación de modelos RF vs LR vs DT|
| 13 | Tuning de n_estimators con GridSearchCV |
| 14 | GridSearch multidimensional con heatmap |
| 15 | Curva de Codo para n_estimators óptimo |
| **16** | **Feature Importances y selección de variables** |

---

## 📚 Referencias

- Breiman, L. (2001). [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324). *Machine Learning*, 45, 5–32.
- Strobl, C., et al. (2007). [Bias in random forest variable importance measures](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-8-25). *BMC Bioinformatics*, 8:25.
- Scikit-learn: [Feature Importances with a Forest of Trees](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html)
- Scikit-learn: [Permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- Géron, A. (2019). *Hands-On Machine Learning*, Cap. 7. O'Reilly.

---

© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).